In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
      os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

import time
!pip install 'kaggle-environments==0.1.6'
import math
import copy
import random

from kaggle_environments import evaluate, make, utils

from collections import deque

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: kaggle-environments
    Found existing installation: kaggle-environments 1.29.3
    Uninstalling kaggle-environments-1.29.3:
      Successfully uninstalled kaggle-environments-1.29.3


In [2]:
import gc

gc.disable()

In [3]:
rollout_times = []
pickBestMove_times = []
chooseMove_times = []
playMove_times = []

In [4]:
env = make("connectx", debug=True)
# env.render() # text format

class Node:
    node_count = 0
    def __init__ (self, board, parent, to_play, first_empty_row_idx=None, bitboards=None) :
        self.board = board
        self.parents = [parent] if parent else []
        self.to_play = to_play
        self.t = 0
        self.n = 0
        self.children = []

        # Create self.column_heights to store the index of the first available row in each column
        self.first_empty_row_idx = []
        if first_empty_row_idx is not None: self.first_empty_row_idx = first_empty_row_idx
        else:
            for column in range(n_columns):
                row = -1 # indicates that the column is full
                while row+1<n_rows and board[(row+1)*n_columns + column] == 0: row = row + 1
                self.first_empty_row_idx.append(row)

        self.untried_moves = [c for c in range(n_columns) if self.first_empty_row_idx[c] >= 0]
        self.bitboards = bitboards if bitboards is not None else list_to_bitboards(board)

        Node.node_count += 1

def list_to_bitboards(board):
    # Initialize empty bitboards for Player 1 and Player 2
    bitboards = [0, 0]
    
    for row in range(n_rows):
        for col in range(n_columns):
            mark = board[row * n_columns + col]
            if mark > 0:
                # Invert row index so the bottom row is bit_row = 0
                bit_row = (n_rows - 1) - row
                
                # Calculate row-major bit position with padding
                bit_position = bit_row * BITS_PER_ROW + col
                bitboards[mark - 1] |= (1 << bit_position)
                
    return bitboards

def choose_max_UCB_child (parent_node):
    """Returns child node with highest UCB score"""
    max_UCB = float('-inf')
    max_UCB_child_node = None
    log_parent_n = math.log(parent_node.n)
    for child_node in parent_node.children:
        if child_node.n == 0: return child_node
        child_ucb = (child_node.t/child_node.n) + 1.414 * (log_parent_n / child_node.n)**0.5
        if child_ucb > max_UCB:
            max_UCB, max_UCB_child_node = child_ucb, child_node
            
    # print("choose_max_UCB_child completed")
    return max_UCB_child_node

def check_win_bitboard(bitboard):
    # If the board doesn't contain enough tokens to make a chain, skip
    if bitboard == 0:
        return False
        
    for direction in DIRECTIONS:
        temp_board = bitboard
        # Repeatedly shift and intersect the bitboard mask
        for _ in range(inarow - 1):
            temp_board &= (temp_board >> direction)
        
        # If any bits survive the cascading intersection filters, a win exists
        if temp_board > 0:
            return True
            
    return False

def terminalStatus (board, bitboards):
    """Returns two elements: isTerminal, and the mark of the winner if it is terminal"""
    cache_key = (bitboards[0], bitboards[1])
    if cache_key in terminal_cache: return terminal_cache[cache_key]
    
    if check_win_bitboard(bitboards[0]): result = [True, 1]
    elif check_win_bitboard(bitboards[1]): result = [True, 2]
    elif 0 not in board[:n_columns]: result = [True, 0]
    else: result = [False, 0]

    terminal_cache[cache_key] = result
    return result

def playMove (board, col, player_mark, parent_empty_rows, bitboards):
    func_start_time = time.process_time()
    new_board = list(board)
    child_empty_rows = list(parent_empty_rows)
    child_bitboards = list(bitboards)

    row = parent_empty_rows[col]
    new_board[row * n_columns + col] = player_mark
    child_empty_rows[col] = row - 1

    bit_row = (n_rows - 1) - row
    bit_position = bit_row * BITS_PER_ROW + col
    child_bitboards[player_mark - 1] |= (1 << bit_position)

    playMove_times.append(time.process_time() - func_start_time)
    
    return new_board, child_empty_rows, child_bitboards

def rollout (node):
    """
    Performs rollout from the input node.
    Returns the mark of the winner found at the end of the rollout.
    """
    func_start_time = time.process_time()
    current_board = node.board
    current_to_play = node.to_play
    current_empty_rows = list(node.first_empty_row_idx)
    current_bitboards = node.bitboards

    rollout_depth = 0

    while True:
        rollout_depth += 1
        if rollout_depth > 50: print(f"    Autobots, there's too much rollout! : {rollout_depth}")
        is_terminal, winner = terminalStatus(current_board, current_bitboards)
        if is_terminal:
            rollout_times.append(time.process_time() - func_start_time)
            return winner
            
        valid_cols = [c for c in range(n_columns) if current_empty_rows[c] >= 0]
        
        if not valid_cols: return 0
            
        col = random.choice(valid_cols)
        current_board, current_empty_rows, current_bitboards = playMove(current_board, col, current_to_play, current_empty_rows, current_bitboards)
        current_to_play = 3 - current_to_play

def backpropagate_vals (start_node, winner_mark):
    """
    Adds val to `t` values of all ancestors in the lineage
    Adds 1 to `n` values of all ancestors in the lineage
    """

    q1 = deque()
    q2 = deque()

    q1.append(start_node)

    while q1 or q2:
        while q1:
            node = q1.popleft()
            if winner_mark == 3-node.to_play: node.t = node.t + 1
            elif winner_mark == 0: node.t = node.t + 0.5
            node.n = node.n + 1
            for parent in node.parents: q2.append(parent)
            
        while q2:
            node = q2.popleft()
            if winner_mark == 3-node.to_play: node.t = node.t + 1
            elif winner_mark == 0: node.t = node.t + 0.5
            node.n = node.n + 1
            for parent in node.parents: q1.append(parent)

def pickBestMove (node):
    """
    Finds child that was visited most number of times
    """
    func_start_time = time.process_time()

    most_visits = -1
    best_child = node.children[0]
    
    for child_node in node.children:
        if child_node.n > most_visits:
            most_visits, best_child = child_node.n, child_node
            # print("Best move updated")
        
    if best_child is None:
        print("No move found - Investigate")
        for idx in range(n_columns):
            if node.first_empty_row_idx[idx] >= 0: return idx
        pickBestMove_times.append(time.process_time() - func_start_time)
        return 0

    for col in range(n_columns):
        if node.first_empty_row_idx[col] != best_child.first_empty_row_idx[col]:
            pickBestMove_times.append(time.process_time() - func_start_time)
            return col

    pickBestMove_times.append(time.process_time() - func_start_time)
    return 0

def chooseMove (root_board, start_time) :
    func_start_time = time.process_time()
    
    global root_node
    root_bitboards = list_to_bitboards(root_board)
    
    root_node = Node(root_board, None, to_play = 1 if root_board.count(1)==root_board.count(2) else 2, bitboards=root_bitboards)
    root_node.n = 1

    n_iters = 0

    while True: 
        if n_iters % 5 == 0 and time.process_time()-start_time >= TIMEOUT:
            break
            
        n_iters = n_iters + 1
        ## print("MC iteration")
        current_node = root_node

        # is current_node a leaf?
        while not current_node.untried_moves and current_node.children :
            ## print("curr node is leaf")
            # yes
            current_node = choose_max_UCB_child(current_node) # Use UCB to go to next best child 
            ## print("curr node updated")
        
        is_terminal, winner = terminalStatus(current_node.board, current_node.bitboards)
        if is_terminal:
            backpropagate_vals(current_node, winner)
            continue

        # current_node is not a leaf now
        
        if n_iters % 5 == 0 and time.process_time()-start_time >= TIMEOUT: break

        if current_node.untried_moves:
            col = current_node.untried_moves.pop(random.randint(0, len(current_node.untried_moves)-1))
            child_board, child_empty_rows, child_bitboards = playMove(current_node.board, col, current_node.to_play, current_node.first_empty_row_idx, current_node.bitboards)
            child_node = Node(child_board, current_node, 3-current_node.to_play, first_empty_row_idx=child_empty_rows, bitboards=child_bitboards)
            
            current_node.children.append(child_node)
            current_node = child_node
            
            ## print("children added to curr node")
            ## print(len(current_node.children))
                
            ## print("now looking at child[0]")
        
            if n_iters % 5 == 0 and time.process_time()-start_time >= TIMEOUT: break

        # current_node has never been visited before - guaranteed
        # perform rollout
        ## print("Going to perform rollout")
        winner_mark = rollout(current_node)
        ## print("performed rollout")
        backpropagate_vals(current_node, winner_mark)
        ## print("performed backprop")

    '''
    print(f"DEBUG: Root node has {len(root_node.children)} children.")
    for i, child in enumerate(root_node.children):
        print(f"Child {i}: visits={child.n}, score={child.t}")
    '''

    # print(f"n_iters = {n_iters}")

    final_move_column = pickBestMove(root_node)
    ## print(f"Chosen final move column = {final_move_column}")

    chooseMove_times.append(time.process_time() - func_start_time)
    
    return final_move_column

def my_agent (observation, configuration) :
    # print("Move triggered")
    start_time = time.process_time()
    # Keep track of the time at which the bot was prompted
    
    global n_columns, n_rows, inarow, player_mark, opp_mark
    global BITS_PER_ROW, total_bits_needed, use_64bit, SHIFT_V, SHIFT_H, SHIFT_D1, SHIFT_D2, DIRECTIONS
    global TIMEOUT
    TIMEOUT = 1.95

    n_columns = configuration.columns
    n_rows = configuration.rows
    inarow = configuration.inarow

    root_board = observation.board
    player_mark = observation.mark
    opp_mark = 3 - player_mark

    BITS_PER_ROW = n_columns + 1 
    total_bits_needed = BITS_PER_ROW * n_rows
    use_64bit = total_bits_needed <= 64
    
    SHIFT_H = 1                       # Horizontal step (1 column over)
    SHIFT_V = BITS_PER_ROW            # Vertical step (1 row up)
    SHIFT_D1 = BITS_PER_ROW + 1       # Diagonal /
    SHIFT_D2 = BITS_PER_ROW - 1       # Diagonal \
    
    DIRECTIONS = [SHIFT_H, SHIFT_V, SHIFT_D1, SHIFT_D2]

    move = chooseMove(root_board, start_time)
    # move simply denotes the column in which to play the move

    if time.process_time() - start_time >= 2: print(">2s TIMEOUT")

    # print("my agent completed")
    return move

'''
env.reset()
# Play as the first agent against default "random" agent.
env.run([my_agent, "random"])
env.render(mode="ipython", width=500, height=450)
'''

'\nenv.reset()\n# Play as the first agent against default "random" agent.\nenv.run([my_agent, "random"])\nenv.render(mode="ipython", width=500, height=450)\n'

In [5]:
##############################################
##############################################
#
#  PLAY A SINGLE GAME AGAINST AN ONLINE BOT
#
##############################################
##############################################


def play_game (player1, player2, bot_idx) :
    # Create a Connect X environment
    env = make("connectx", debug=True)

    global terminal_cache
    terminal_cache = {}
    terminal_cache.clear()
    
    # Run a single game
    env.run([player1, player2])

    reward = -1
    
    final_step = env.steps[-1]
    if final_step[0]['status'] == 'DONE':
        # Determine the result for your agent (player 1)
        # The reward for the agent is at index 0
        reward = final_step[bot_idx]['reward']
    else:
        reward = -1
        print("The game has not concluded yet.")
    
    # Render the final state of the board and the game replay
    # env.render(mode="ipython")

    return reward
    
##############################################
##############################################
#
#                 TESTER
#
##############################################
##############################################

NUM_GAMES = 10

# BOT PLAYS FIRST
bot_wins = 0
bot_losses = 0
draws = 0
test_start_time = time.time()
FIRST_GAMES = int(math.ceil(NUM_GAMES / 2))

print("BOT PLAYS FIRST")
for test_num in range (FIRST_GAMES) :
    print(f"Playing game {test_num+1} / {FIRST_GAMES}")
    reward = play_game(my_agent, "negamax", 0)

    if reward == 1:
        bot_wins += 1
        # print("BOT WON")
    elif reward == -1:
        bot_losses += 1
        # print("BOT LOST")
    else:  # reward == 0 or None
        draws += 1
        # print("DRAW")

print(f"WINS          = {bot_wins}")
print(f"LOSSES        = {bot_losses}")
print(f"DRAWS         = {draws}")
print(f"TOTAL GAMES   = {FIRST_GAMES}")
print(f"AVG TIME/GAME = {(time.time() - test_start_time)/FIRST_GAMES}")


# BOT PLAYS SECOND
bot_wins = 0
bot_losses = 0
draws = 0
test_start_time = time.time()
SECOND_GAMES = int(math.floor(NUM_GAMES / 2))

print("BOT PLAYS SECOND")
for test_num in range (SECOND_GAMES) :
    print(f"Playing game {test_num+1} / {SECOND_GAMES}")
    reward = play_game("negamax", my_agent, 1)

    if reward == 1:
        bot_wins += 1
        # print("BOT WON")
    elif reward == -1:
        bot_losses += 1
        # print("BOT LOST")
    else:  # reward == 0 or None
        draws += 1
        # print("DRAW")

print(f"WINS          = {bot_wins}")
print(f"LOSSES        = {bot_losses}")
print(f"DRAWS         = {draws}")
print(f"TOTAL GAMES   = {SECOND_GAMES}")
print(f"AVG TIME/GAME = {(time.time() - test_start_time)/SECOND_GAMES}")

BOT PLAYS FIRST
Playing game 1 / 5
Playing game 2 / 5
Playing game 3 / 5
Playing game 4 / 5
Playing game 5 / 5
WINS          = 5
LOSSES        = 0
DRAWS         = 0
TOTAL GAMES   = 5
AVG TIME/GAME = 20.90584807395935
BOT PLAYS SECOND
Playing game 1 / 5
Playing game 2 / 5
Playing game 3 / 5
Playing game 4 / 5
Playing game 5 / 5
WINS          = 5
LOSSES        = 0
DRAWS         = 0
TOTAL GAMES   = 5
AVG TIME/GAME = 17.112536334991454


# TIMING ANALYSIS

In [6]:
print(np.median(rollout_times))
print(np.median(playMove_times))
print(np.median(pickBestMove_times))
print(np.median(chooseMove_times))

7.058699999618057e-05
2.2490000048946968e-06
5.467499999411984e-06
1.9501342950000016


In [7]:
print(np.max(rollout_times))
print(np.max(playMove_times))
print(np.max(pickBestMove_times))
print(np.max(chooseMove_times))

0.06761552699998674
0.0009732620000022507
9.08399999843823e-06
1.9508029019999924


In [8]:
print(sorted(chooseMove_times)[-5:])
print(sorted(rollout_times)[-5:])

[1.9505804359999956, 1.9506863979999878, 1.9507170400000007, 1.9507672780000007, 1.9508029019999924]
[0.05702386799998749, 0.05848180599998898, 0.05917435599999976, 0.06308009699999673, 0.06761552699998674]
